In [1]:
import cv2
import numpy as np
import glob
import os

results_dict = {}

def process_image(image_path):
    img = cv2.imread(image_path)

    if img is None:
        print(f"Warning: Could not read image {image_path}")
        return

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    _, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        print(f"Warning: No contours found in {image_path}")
        return

    largest_contour = max(contours, key=cv2.contourArea)

    (x, y), radius = cv2.minEnclosingCircle(largest_contour)

    filename = os.path.basename(image_path)
    results_dict[filename] = {
        'center': (int(x), int(y)),
        'radius': int(radius)
    }

def main():
    image_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level2\images_for_classification_grade"

    for image_path in glob.glob(os.path.join(image_dir, "*.png")):
        process_image(image_path)


if __name__ == "__main__":
    main()

In [3]:
# import math
# 
# def determine_region(x: int, y: int, circle_data: dict) -> int:
# 
#     center_x, center_y = circle_data['center']
#     radius = circle_data['radius']
#     
#     dx = x - center_x
#     dy = center_y - y  
#     distance = math.sqrt(dx**2 + dy**2)
#     
#     if distance <= radius / 2:
#         return 5
#     
#     if distance > radius:
#         return -1
#     
#     angle = math.degrees(math.atan2(dy, dx)) % 360
#     
#     if 45 <= angle < 135:
#         return 2  
#     elif 135 <= angle < 225:
#         return 3  
#     elif 225 <= angle < 315:
#         return 4  
#     else:
#         return 1  
import math

def determine_region(
    x: int, 
    y: int, 
    circle_data: dict, 
    central_radius_factor: float
) -> int:
    
    center_x, center_y = circle_data['center']
    main_radius = circle_data['radius']
    
    # محاسبه شعاع ناحیه مرکزی
    central_radius = main_radius * central_radius_factor
    
    dx = x - center_x
    dy = center_y - y  # معکوس کردن جهت محور Y
    distance = math.sqrt(dx**2 + dy**2)
    
    # بررسی ناحیه مرکزی با شعاع تنظیم شده
    if distance <= central_radius:
        return 5
    
    # بررسی خارج بودن از دایره اصلی
    if distance > main_radius:
        return -1
    
    # محاسبه زاویه
    angle = math.degrees(math.atan2(dy, dx)) % 360
    
    # تشخیص ناحیه بر اساس زاویه
    if 45 <= angle < 135:
        return 2
    elif 135 <= angle < 225:
        return 3
    elif 225 <= angle < 315:
        return 4
    else:
        return 1

In [4]:
def process_images(image_dir, circle_data_dict):
    results = {}
    
    for image_path in glob.glob(os.path.join(image_dir, "*.png")):
        filename = os.path.basename(image_path)
        
        if filename not in circle_data_dict:
            continue
            
        img = cv2.imread(image_path)
        if img is None:
            continue
            
        # تبدیل به فضای رنگی HSV برای تشخیص بهتر رنگ قرمز
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        
        # محدوده رنگی قرمز در HSV
        lower_red1 = np.array([0, 120, 70])
        upper_red1 = np.array([10, 255, 255])
        lower_red2 = np.array([170, 120, 70])
        upper_red2 = np.array([180, 255, 255])
        
        # ایجاد ماسک برای رنگ قرمز
        mask_red1 = cv2.inRange(hsv, lower_red1, upper_red1)
        mask_red2 = cv2.inRange(hsv, lower_red2, upper_red2)
        red_mask = cv2.bitwise_or(mask_red1, mask_red2)
        
        # پیش‌پردازش ماسک
        kernel = np.ones((3,3), np.uint8)
        red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_CLOSE, kernel)
        red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_OPEN, kernel)
        
        # پیدا کردن کانتورها روی ماسک قرمز
        contours, _ = cv2.findContours(red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        regions = set()
        
        for cnt in contours:
            if cv2.contourArea(cnt) < 50:
                continue
                
            for point in cnt[:,0,:]:
                x, y = point
                region = determine_region(x, y, circle_data_dict[filename], 0.30)
                if region != -1:
                    regions.add(region)
        
        results[filename] = sorted(list(regions)) if regions else []
    
    return results

image_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level2\images_for_classification_grade"
final_results = process_images(image_dir, results_dict)
def classify_regions(final_results):
    classified = {}
    for filename, regions in final_results.items():
        if 5 in regions:
            classified[filename] = 4
        else:
            num_regions = len(regions)
            if num_regions == 0:
                classified[filename] = 0
            elif num_regions == 1:
                classified[filename] = 1
            elif num_regions == 2:
                classified[filename] = 2
            else: 
                classified[filename] = 3
    return classified

classified_results = classify_regions(final_results)
classified_results

{'359.png': 4,
 '360.png': 4,
 '361.png': 4,
 '362.png': 4,
 '363.png': 2,
 '364.png': 4,
 '365.png': 4,
 '366.png': 3,
 '367.png': 4,
 '368.png': 4,
 '369.png': 4,
 '370.png': 1,
 '371.png': 1,
 '372.png': 2,
 '373.png': 1,
 '374.png': 2,
 '375.png': 4,
 '376.png': 4,
 '377.png': 4,
 '378.png': 4,
 '379.png': 4,
 '380.png': 4,
 '381.png': 4,
 '382.png': 3,
 '383.png': 1,
 '384.png': 3,
 '385.png': 4,
 '386.png': 2,
 '387.png': 1,
 '388.png': 0,
 '389.png': 1,
 '390.png': 4,
 '391.png': 1,
 '392.png': 1,
 '393.png': 2,
 '394.png': 4,
 '395.png': 2,
 '396.png': 0,
 '397.png': 1,
 '398.png': 0,
 '399.png': 4,
 '400.png': 0,
 '401.png': 1,
 '402.png': 1,
 '403.png': 1,
 '404.png': 1,
 '405.png': 2,
 '406.png': 4,
 '407.png': 1,
 '408.png': 4,
 '409.png': 4,
 '410.png': 2,
 '411.png': 1,
 '412.png': 1,
 '413.png': 1,
 '414.png': 1,
 '415.png': 1,
 '416.png': 4,
 '417.png': 1,
 '418.png': 2,
 '419.png': 1,
 '420.png': 1,
 '421.png': 1,
 '422.png': 4,
 '423.png': 4,
 '424.png': 4,
 '425.png'

In [6]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    confusion_matrix,
    balanced_accuracy_score
)

label_path = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level2\label.xlsx"

df = pd.read_excel(label_path)
df['numeric_id'] = df.iloc[:, 0].str.extract('(\d+)').astype(int)  
filtered_df = df[df['numeric_id'] >= 359]

filtered_df['filename_key'] = filtered_df['numeric_id'].astype(str) + '.png'

y_true = filtered_df['Type'].values
y_pred = [classified_results.get(key, 0) for key in filtered_df['filename_key']]

accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='macro')
balanced_acc = balanced_accuracy_score(y_true, y_pred)

cm = confusion_matrix(y_true, y_pred)
sensitivity = recall_score(y_true, y_pred, average='macro')

specificities = []
for i in range(cm.shape[0]):
    tn = cm.sum() - cm[i,:].sum() - cm[:,i].sum() + cm[i,i]
    fp = cm[:,i].sum() - cm[i,i]
    specificities.append(tn / (tn + fp) if (tn + fp) != 0 else 0)
specificity = np.mean(specificities)

print(f"""
=================================
 (Accuracy): {accuracy:.4f}
 F1 (Macro): {f1:.4f}
 (Sensitivity): {sensitivity:.4f}
 (Specificity): {specificity:.4f}
دقت  (Balanced Accuracy): {balanced_acc:.4f}
=================================
""")


 (Accuracy): 0.6271
 F1 (Macro): 0.2742
 (Sensitivity): 0.3266
 (Specificity): 0.8977
دقت  (Balanced Accuracy): 0.4083



C:\Users\Meta Pc\AppData\Local\Temp\ipykernel_2252\190139932.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['filename_key'] = filtered_df['numeric_id'].astype(str) + '.png'
C:\Users\Meta Pc\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:2394: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
C:\Users\Meta Pc\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [7]:
c = 359
for i in range(len(y_pred)):
    print(c, y_true[i], y_pred[i])
    c += 1

# y_true
# y_pred

359 4 4
360 4 4
361 4 4
362 4 4
363 4 2
364 4 4
365 4 4
366 4 3
367 4 4
368 4 4
369 4 4
370 2 1
371 4 1
372 1 2
373 4 1
374 4 2
375 2 4
376 4 4
377 4 4
378 4 4
379 4 4
380 4 4
381 4 4
382 2 3
383 2 1
384 3 3
385 4 4
386 4 2
387 1 1
388 1 0
389 4 1
390 4 4
391 1 1
392 1 1
393 4 2
394 4 4
395 4 2
396 2 0
397 2 1
398 4 0
399 4 4
400 1 0
401 4 1
402 1 1
403 1 1
404 2 1
405 4 2
406 4 4
407 1 1
408 4 4
409 4 4
410 4 2
411 4 1
412 4 1
413 4 1
414 1 1
415 4 1
416 4 4
417 1 1
418 4 2
419 2 1
420 1 1
421 4 1
422 4 4
423 4 4
424 4 4
425 4 4
426 4 3
427 4 4
428 4 4
429 4 4
430 4 4
431 4 4
432 4 3
433 4 4
434 4 4
435 4 4
436 4 4
437 4 4
438 4 1
439 4 4
440 4 2
441 4 4
442 4 4
443 4 4
444 4 4
445 4 4
446 4 2
447 1 1
448 4 4
449 4 2
450 4 4
451 4 4
452 2 4
453 4 4
454 1 2
455 4 3
456 4 4
457 4 4
458 4 4
459 4 4
460 4 4
461 1 3
462 1 4
463 1 0
464 4 4
465 4 4
466 4 1
467 4 4
468 4 4
469 4 4
470 4 4
471 4 2
472 4 3
473 4 3
474 4 3
475 4 4
476 2 4
477 4 3
478 4 4
479 4 2
480 4 3
481 4 4
482 3 1
483 4 4


just for visualization

In [21]:
import cv2
import numpy as np
import math
import glob
import os

REGION_COLORS = {
    1: (0, 0, 255, 0.3),    # قرمز برای ناحیه 1
    2: (0, 255, 0, 0.3),    # سبز برای ناحیه 2
    3: (255, 0, 0, 0.3),    # آبی برای ناحیه 3
    4: (0, 255, 255, 0.3),  # زرد برای ناحیه 4
    5: (255, 0, 255, 0.3)   # بنفش برای ناحیه 5
}

def visualize_regions(
    image_path, 
    circle_data, 
    output_dir, 
    central_radius_factor=0.3  # پارامتر جدید برای تنظیم اندازه ناحیه مرکزی
):
    """تابع بصری‌سازی با قابلیت تنظیم اندازه ناحیه مرکزی"""
    
    # خواندن تصویر اصلی
    img = cv2.imread(image_path)
    if img is None:
        return
    
    # ایجاد یک کپی برای بصری‌سازی
    vis_img = img.copy()
    h, w = img.shape[:2]
    
    # استخراج اطلاعات دایره
    center = circle_data['center']
    main_radius = circle_data['radius']
    cx, cy = center
    
    # محاسبه شعاع ناحیه مرکزی
    central_radius = main_radius * central_radius_factor
    
    # ایجاد شبکه مختصات
    y_coords, x_coords = np.indices((h, w))
    
    # محاسبه مختصات نسبی
    dx = x_coords - cx
    dy = cy - y_coords  # معکوس کردن جهت محور Y
    
    # محاسبه فاصله و زاویه
    distance = np.sqrt(dx**2 + dy**2)
    angle = np.degrees(np.arctan2(dy, dx)) % 360
    
    # ایجاد ماسک برای هر ناحیه با شعاع تنظیم شده
    masks = {
        5: (distance <= central_radius),
        1: (distance > central_radius) & (distance <= main_radius) & ((angle < 45) | (angle >= 315)),
        2: (distance > central_radius) & (distance <= main_radius) & (angle >= 45) & (angle < 135),
        3: (distance > central_radius) & (distance <= main_radius) & (angle >= 135) & (angle < 225),
        4: (distance > central_radius) & (distance <= main_radius) & (angle >= 225) & (angle < 315)
    }
    
    # اضافه کردن رنگ‌ها به تصویر
    for region, mask in masks.items():
        color = REGION_COLORS[region][:3]
        alpha = REGION_COLORS[region][3]
        overlay = np.zeros_like(vis_img)
        overlay[mask] = color
        vis_img = cv2.addWeighted(vis_img, 1, overlay, alpha, 0)
    
    # کشیدن مرزها و خطوط با شعاع تنظیم شده
    cv2.circle(vis_img, center, main_radius, (0,0,0), 2)           # دایره اصلی
    cv2.circle(vis_img, center, int(central_radius), (0,0,0), 2)   # دایره مرکزی تنظیم شده
    
    # کشیدن خطوط تقسیم‌کننده
    for ang in [45, 135, 225, 315]:
        x2 = int(cx + main_radius * math.cos(math.radians(ang)))
        y2 = int(cy - main_radius * math.sin(math.radians(ang)))
        cv2.line(vis_img, center, (x2, y2), (0,0,0), 2)
    
    # ذخیره نتیجه
    output_path = os.path.join(output_dir, os.path.basename(image_path))
    cv2.imwrite(output_path, vis_img)

# تنظیمات اجرایی
image_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level2\images_for_classification_grade"
output_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level2\region"
os.makedirs(output_dir, exist_ok=True)

# تنظیم ضریب دلخواه (مثال: 0.7)
DESIRED_CENTRAL_FACTOR = 0.25

# پردازش تمام تصاویر با ضریب تنظیم شده
for image_path in glob.glob(os.path.join(image_dir, "*.png")):
    filename = os.path.basename(image_path)
    if filename in results_dict:
        visualize_regions(
            image_path=image_path,
            circle_data=results_dict[filename],
            output_dir=output_dir,
            central_radius_factor=DESIRED_CENTRAL_FACTOR  # ارسال پارامتر تنظیمی
        )

print(f"تصاویر با ضریب ناحیه مرکزی {DESIRED_CENTRAL_FACTOR} در مسیر {output_dir} ذخیره شدند.")

تصاویر با ضریب ناحیه مرکزی 0.25 در مسیر C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level2\region ذخیره شدند.
